In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surfaceomeTopography import cell_surface, mean_height, plot_surfaces
%matplotlib inline

In [ ]:
surfaceome = pd.read_csv(str(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab')), sep = '\t')

In [ ]:
surfaceome['ID link'] = surfaceome.apply(lambda row: row['ID link'].split(','), axis = 1)
surfaceome = surfaceome.explode('ID link')

In [ ]:
expression = pd.read_csv(str(resolve('inputs/expression/ravenhill_2020_s1c_monocyte_abundance.csv')))

In [ ]:
surf = pd.merge(surfaceome, expression, left_on = 'ID link', right_on = 'Uniprot-s', how = 'inner')

In [ ]:
surf.drop_duplicates(subset = 'ID link', keep = 'last', inplace = True)

In [ ]:
heights = pd.read_csv(str(resolve('database/height_estimates.csv')))

#heights.drop('Unnamed: 0', axis =1, inplace = True)
heights.rename(columns = {"ID link_first":"ID link",
                         "Entry name_first": "Entry name",
                         "dom_len_count_domain": "domain counts"}, inplace = True)
#surf_h = pd.merge(surf.drop_duplicates(subset = ['Gene ID']), heights, on = 'ID link')


In [ ]:
surf_h = pd.merge(surf, heights, on = 'ID link')

In [ ]:
surf_h.to_csv(out_path('tables/ravenhill_2020_monocyte_expression_dataset.csv'))

In [ ]:
#The surface of each monocyte subset, from the median of its three donors
keep = ['total_height', 'ID link', 'Gene names']
classical_donors = [f'Classical monocytes, donation {i}' for i in (1, 2, 3)]
intermediate_donors = [f'Intermediate monocytes, donation {i}' for i in (1, 2, 3)]
non_classical_donors = [f'Non-classical monocytes, donation {i}' for i in (1, 2, 3)]

classical_monocytes = cell_surface(surf_h, classical_donors, columns=classical_donors + keep)
intermediate_monocytes = cell_surface(surf_h, intermediate_donors, columns=intermediate_donors + keep)
non_classical = cell_surface(surf_h, non_classical_donors, columns=non_classical_donors + keep)

In [ ]:
#expression-weighted mean height of each subset, nm
monocyte_subsets = {'Classical Monocytes': classical_monocytes,
                    'intermediate monocytes': intermediate_monocytes,
                    'non-classical': non_classical}
pd.Series({name: mean_height(surface) for name, surface in monocyte_subsets.items()}).round(2)

In [ ]:
#histogram on linear scale
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
#plt.rc('font', **font)

plt.figure(figsize=(7,7))

 
x = classical_monocytes['total_height']
w = classical_monocytes['percent_expression']


plt.subplot(211)
hist, bins, _ = plt.hist(x, bins=30, alpha = 0.5, weights = w)
#plt.yscale('log')

# histogram on log scale. 
# Use non-equal bin sizes, such that they look equal on log scale.
logbins = np.logspace(np.log10(bins[1]),np.log10(bins[-1]),len(bins))
plt.subplot(212)
plt.hist(x, bins=logbins, alpha = 0.5, weights = w)



plt.xscale('log')
plt.xlabel('height (nm)')
plt.legend(['classical monocyte donation 2'])

plt.show()

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

plot_surfaces(monocyte_subsets).savefig(out_path('figures/F4_monocyte_subtypes.pdf'), transparent=True)

In [ ]:
plot_surfaces(monocyte_subsets, overlay=True).savefig(out_path('figures/F4_monocyte_subtypes_overlap.pdf'),
                                                   transparent=True)

In [ ]:
comparison = pd.merge(non_classical,
                     classical_monocytes,
                     on = ['ID link', 'Gene names', 'total_height',
                          ], 
                      how = 'outer', suffixes = ('_non', '_classical'))

In [ ]:
comparison_int = pd.merge(intermediate_monocytes,
                     classical_monocytes,
                     on = ['ID link', 'Gene names', 'total_height',
                          ], 
                      how = 'outer', suffixes = ('_int', '_classical'))

In [ ]:
comparison_non_int = pd.merge(non_classical,
                     intermediate_monocytes,
                     on = ['ID link', 'Gene names', 'total_height',
                          ], 
                      how = 'outer', suffixes = ('_non', '_int'))

In [ ]:
import scipy.stats

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests, fdrcorrection

comparison= comparison.assign(fractional_difference = (comparison['percent_expression_non']- comparison['percent_expression_classical'])/comparison['percent_expression_classical'])
comparison= comparison.assign(ratio = (comparison['percent_expression_non'])/comparison['percent_expression_classical'])
#comparison= comparison.assign(n = 3)

comparison= comparison.assign(p_val = pd.Series(scipy.stats.ttest_ind(comparison[['Classical monocytes, donation 1', 
                                                                       'Classical monocytes, donation 2',
                                                                       'Classical monocytes, donation 3']], 
                                                            comparison[['Non-classical monocytes, donation 1', 
                                                                       'Non-classical monocytes, donation 2',
                                                                       'Non-classical monocytes, donation 3']], axis = 1)[1]))
# Extract the p-values column
p_values = comparison['p_val'].dropna()

# Choose a correction method (e.g., Benjamini-Hochberg)
#method = 'fdr_tsbh'  # BH (Benjamini-Hochberg) correction

# Perform multiple testing correction
rejected, corrected_p_values = fdrcorrection(p_values, alpha=0.1, method='p', is_sorted=False)


comparison = comparison.assign(sig = np.where(comparison['p_val'] < 0.05, True, False))

# comparison[((comparison['percent_expression_non'] > 1)|(comparison['percent_expression_classical'] > 1))&((comparison['fractional_difference']>0.25)|(comparison['fractional_difference']<-0.25))& (comparison['p_val'] < 0.05)].sort_values('fractional_difference')

In [ ]:
comparison_int= comparison_int.assign(fractional_difference = (comparison_int['percent_expression_int']- comparison_int['percent_expression_classical'])/comparison_int['percent_expression_classical'])
comparison_int= comparison_int.assign(ratio = (comparison_int['percent_expression_int'])/comparison_int['percent_expression_classical'])
#comparison_int= comparison_int.assign(n = 3)

comparison_int= comparison_int.assign(p_val = pd.Series(scipy.stats.ttest_ind(comparison_int[['Classical monocytes, donation 1', 
                                                                       'Classical monocytes, donation 2',
                                                                       'Classical monocytes, donation 3']], 
                                                            comparison_int[['Intermediate monocytes, donation 1', 
                                                                       'Intermediate monocytes, donation 2',
                                                                       'Intermediate monocytes, donation 3']], axis = 1)[1]))

# Extract the p-values column
p_values = comparison['p_val'].dropna()

# Choose a correction method (e.g., Benjamini-Hochberg)
#method = 'fdr_tsbh'  # BH (Benjamini-Hochberg) correction

# Perform multiple testing correction
rejected, corrected_p_values = fdrcorrection(p_values, alpha=0.1, method='p', is_sorted=False)


comparison_int= comparison_int.assign(sig = np.where(comparison_int['p_val'] < 0.05, True, False))

comparison_int[((comparison_int['percent_expression_int'] > 1)|(comparison_int['percent_expression_classical'] > 1))&((comparison_int['fractional_difference']>0.25)|(comparison_int['fractional_difference']<-0.25))& (comparison_int['p_val'] < 0.05)].sort_values('fractional_difference')

In [ ]:


comparison_non_int= comparison_non_int.assign(fractional_difference = (comparison_non_int['percent_expression_non']- comparison_non_int['percent_expression_int'])/comparison_non_int['percent_expression_int'])
comparison_non_int= comparison_non_int.assign(ratio = (comparison_non_int['percent_expression_non'])/comparison_non_int['percent_expression_int'])
#comparison_non_int= comparison_non_int.assign(n = 3)

comparison_non_int= comparison_non_int.assign(p_val = pd.Series(scipy.stats.ttest_ind(comparison_non_int[['Non-classical monocytes, donation 1', 
                                                                       'Non-classical monocytes, donation 2',
                                                                       'Non-classical monocytes, donation 3']], 
                                                            comparison_non_int[['Intermediate monocytes, donation 1', 
                                                                       'Intermediate monocytes, donation 2',
                                                                       'Intermediate monocytes, donation 3']], axis = 1)[1]))

# reference note pasted from the statsmodels docs, not executable:
# statsmodels.stats.multitest.multipletests(pvals, alpha=0.05, method='hs', maxiter=1, is_sorted=False, returnsorted=False)

comparison_non_int= comparison_non_int.assign(sig = np.where(comparison_non_int['p_val'] < 0.05, True, False))

comparison_non_int[((comparison_non_int['percent_expression_non'] > 1)|(comparison_non_int['percent_expression_int'] > 1))&((comparison_non_int['fractional_difference']>0.25)|(comparison_non_int['fractional_difference']<-0.25))& (comparison_non_int['p_val'] < 0.05)].sort_values('fractional_difference')

In [ ]:
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
sns.set(style="white")

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison, x = 'total_height', y = 'ratio', hue = 'sig', size = 'percent_expression_classical',
                sizes=(20, 300), legend = 'brief', palette = 'Paired')

ax.axhline(y= 1)
ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)
plt.xlim(1, 400)
plt.ylim(0.05, 15)
plt.yscale('log')
plt.xscale('log')


fig.savefig(out_path('figures/F4_expression_ratio_vs_size_non_vs_classical.pdf'))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison_int, x = 'total_height', y = 'ratio', hue = 'sig', size = 'percent_expression_int',
                sizes=(20, 300), legend = 'brief', palette = 'Paired')

ax.axhline(y= 1)
ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)
plt.xlim(0.1, 1400)
plt.ylim(0.05, 15)
plt.yscale('log')
plt.xscale('log')


fig.savefig(out_path('figures/F4_expression_ratio_vs_size_intermediate_vs_classical.pdf'))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison_non_int, x = 'total_height', y = 'ratio', hue = 'sig', size = 'percent_expression_int',
                sizes=(20, 300), legend = 'brief', palette = 'Paired')

ax.axhline(y= 1)
ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)
plt.xlim(0.1, 1400)
#plt.ylim(0.001, 1000)
plt.ylim(0.05, 15)
plt.yscale('log')
plt.xscale('log')


fig.savefig(out_path('figures/F4_expression_ratio_vs_size_intermediate_vs_non_classical.pdf'))

In [ ]:
import plotly.express as px
fig = px.scatter(comparison, x = 'total_height', y= 'ratio', color = 'sig', size = 'percent_expression_classical',
                hover_name= 'Gene names', hover_data = ['percent_expression_classical', 'percent_expression_non'])

ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)

fig.update_layout(xaxis_type="log", yaxis_type="log")

fig.show()

In [ ]:
import plotly.express as px
fig = px.scatter(comparison_int, x = 'total_height', y= 'ratio', color = 'sig', size = 'percent_expression_int',
                hover_name= 'Gene names', hover_data = ['percent_expression_classical', 'percent_expression_int'])

ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)

fig.update_layout(xaxis_type="log", yaxis_type="log")

fig.show()

In [ ]:
import plotly.express as px
fig = px.scatter(comparison_non_int, x = 'total_height', y= 'ratio', color = 'sig', size = 'percent_expression_int',
                hover_name= 'Gene names', hover_data = ['percent_expression_int', 'percent_expression_non'])

ax.axhline(y= 0.5, color = 'r', alpha = 0.5)
ax.axhline(y= 2, color = 'r', alpha = 0.5)

fig.update_layout(xaxis_type="log", yaxis_type="log")

fig.show()

In [ ]:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison, x = 'ratio', y = 'p_val', hue = 'sig', legend = False, palette = 'Paired', size = 'percent_expression_non')
ax.axvline(x= 1)
ax.axvline(x= 0.5, color = 'r', alpha = 0.5)
ax.axvline(x= 2, color = 'r', alpha = 0.5)
ax.axhline(y = 0.05, color = 'c')
plt.xscale('log', base = 2)
plt.yscale('log')
plt.ylim(0.00001, 1)
plt.xlim(0.06, 16)
ax.invert_yaxis()

fig.savefig(out_path('figures/F4_expression_ratio_vs_pval_non_vs_classical.pdf'))

In [ ]:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison_int, x = 'ratio', y = 'p_val', hue = 'sig', legend = False, palette = 'Paired', size = 'percent_expression_int')
ax.axvline(x= 1)
ax.axvline(x= 0.5, color = 'r', alpha = 0.5)
ax.axvline(x= 2, color = 'r', alpha = 0.5)
ax.axhline(y = 0.05, color = 'c')
plt.xscale('log', base = 2)
plt.yscale('log')
plt.ylim(0.00001, 1)
plt.xlim(0.06, 16)
ax.invert_yaxis()

fig.savefig(out_path('figures/F4_expression_ratio_vs_pval_intermediate_vs_classical.pdf'))

In [ ]:
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

fig, ax = plt.subplots(figsize=(7, 4))

sns.scatterplot(data = comparison_non_int, x = 'ratio', y = 'p_val', hue = 'sig', legend = False, palette = 'Paired', size = 'percent_expression_int')
ax.axvline(x= 1)
ax.axvline(x= 0.5, color = 'r', alpha = 0.5)
ax.axvline(x= 2, color = 'r', alpha = 0.5)
ax.axhline(y = 0.05, color = 'c')
plt.xscale('log', base = 2)
plt.yscale('log')
plt.ylim(0.00001, 1)
plt.xlim(0.06, 16)
ax.invert_yaxis()

fig.savefig(out_path('figures/F4_expression_ratio_vs_pval_intermediate_vs_non_classical.pdf'))

In [ ]:
p_val = scipy.stats.ttest_ind(comparison[['Classical monocytes, donation 1', 
                                                                       'Classical monocytes, donation 2',
                                                                       'Classical monocytes, donation 3']], 
                                                            comparison[['Non-classical monocytes, donation 1', 
                                                                       'Non-classical monocytes, donation 2',
                                                                       'Non-classical monocytes, donation 3']], axis = 1)